This notebook studies lateral load sensitivity. That is, the effects that slip angle and load force have on lateral force.

In [1]:
from datetime import datetime

import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(suppress=True, precision=5) # Disable scientific notation

SELECTED_RUNS = [4, 5, 6, 8, 9]

PACEJKA_PARAMS_NAMES = ["B", "C", "D", "E", "F"]
PACEJKA_PARAMS_GUESS = [-0.1, 0.1, 2000, 0.3, 0]  # Update as needed

def quad_fit(Fz, a0, a1, a2):
    return a0 + a1 * Fz + a2 * (Fz ** 2)

def lin_fit(Fz, a0, a1):
    return a0 + Fz * a1

def exp_fit(Fz, a0, a1, a2):
    return a0 + a1 * (np.e ** (a2 * Fz))

PACEJKA_PARAMS_FIT_FNS = [quad_fit, quad_fit, lin_fit, exp_fit, quad_fit]
PACEJKA_PARAMS_FIT_GUESS = [
    [-0.37, -0.00045, -0.0000002],
    [0, 0, 0],
    [0, 0],
    [0, 0, 0],
    [0, 0, 0]
]

In [2]:
def get_run_data(selected_runs):
    metric_data = pd.DataFrame()

    for rn in selected_runs:
        # Import the data
        metric_datum = pd.read_csv(f'./SN5_R9_Lateral/run{rn}.csv')

        metric_datum["ET"] = metric_datum["ET"] - metric_datum["ET"].iloc[0]
        
        if len(metric_data) != 0:
            metric_datum["ET"] = metric_datum["ET"] + metric_data["ET"].iloc[-1]

        metric_data = pd.concat([metric_data, metric_datum])
        print(f"Loaded file run{rn}.dat ({len(metric_datum)} rows)")

    return metric_data

def bin_by_param(df, bins, param_col):
    out = []
    for lb, ub in bins:
        sub = df[(df[param_col] >= lb) & (df[param_col] < ub)]
        out.append(sub)
    return out

In [3]:
def pacejka_lateral(alpha, B, C, D, E, F):
    return D * np.sin(C * np.arctan(B * alpha - E * (B * alpha - np.arctan(B * alpha))) + F)

def pacejka_lateral_with_parameter(alpha, Fz, hyper_coefficients):
    B, C, D, E, F = [PACEJKA_PARAMS_FIT_FNS[i](Fz, *hyper_coefficients[i]) for i in range(5)]

    return D * np.sin(C * np.arctan(B * alpha - E * (B * alpha - np.arctan(B * alpha))) + F)

def pacejka_lateral_derivative_with_parameter(alpha, Fz, hyper_coefficients):
        B, C, D, E, F = [PACEJKA_PARAM_FIT_FNS[i](Fz, *hyper_coefficients[i]) for i in range(5)]

        return -(C * D * (E * (B - B / ((B ** 2) * (alpha ** 2) + 1)) - B) * np.cos(C * np.arctan(E * (B * alpha - np.arctan(B * alpha)) - B * alpha) - F)) / (((E * (B * alpha - np.arctan(B * alpha)) - B * alpha) ** 2) + 1)

def fit_pacejka(df, x_col, y_col):
    popt, _ = curve_fit(
        pacejka_lateral, 
        df[x_col], 
        df[y_col], 
        p0=PACEJKA_PARAMS_GUESS, 
        maxfev=50000
    )

    return popt

In [4]:
all_data = get_run_data(SELECTED_RUNS)

Loaded file run4.dat (36194 rows)
Loaded file run5.dat (31138 rows)
Loaded file run6.dat (52331 rows)
Loaded file run8.dat (61020 rows)
Loaded file run9.dat (52322 rows)


In [5]:
Fz_bins = [(-1300, -1000), (-1000, -750), (-750, -500), (-500, -300), (-300, -150)] # Empirically found by looking at graph.
all_data_binned = bin_by_param(all_data, Fz_bins, "FZ")

params_binned = np.zeros((len(Fz_bins), 5))
for i in range(5):
    data_bin = all_data_binned[i]
    params_binned[i, :] = fit_pacejka(data_bin, "SA", "FY")
    print(f"Fitted bin {i + 1}")

Fitted bin 1
Fitted bin 2
Fitted bin 3
Fitted bin 4
Fitted bin 5


In [6]:
def surface_fig(all_data, hyper_coefficients, Fz_bins, slip_angle, load_force):

    Fz_domain = np.linspace(all_data["FZ"].min(), -225, 100)
    SA_domain = np.linspace(all_data["SA"].min(), all_data["SA"].max(), 100)

    Fy_surface = np.zeros(shape=(len(Fz_domain), len(SA_domain)))
    for i in range(len(Fz_domain)):
        for j in range(len(SA_domain)):
            Fy_surface[j][i] = pacejka_lateral_with_parameter(SA_domain[j], Fz_domain[i], hyper_coefficients)

    fig = go.Figure(data=[go.Surface(
        x=Fz_domain,
        y=SA_domain,
        z=Fy_surface
    )])

    # fig.add_trace(go.Scatter3d(
      #   x=[load_force], 
        # y=[slip_angle],
        # z=[pacejka_lateral_with_parameter(slip_angle, load_force, hyper_coefficients)],
        # marker=dict(
   #          color="white"
     #   )
    #))

    CA_YELLOW = "#FDB515"
    BK_BLUE = "#002676"

    LABEL_COLOR = BK_BLUE
    TICK_COLOR = CA_YELLOW
    
    SCENE_BACKGROUND = "white" #"Greenscreen", to photoshop out background
    PLANE_COLOR = "#242424"

    fig.update_layout(
        scene=dict(
            bgcolor=SCENE_BACKGROUND,
            xaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Load Force (N)', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
            yaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Slip Angle (deg)', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
            zaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Fy (N)', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
        ),
        title=dict(
            x=0.5,
            xanchor='center',
            text='Instantaneous Cornering Stiffness',
            font=dict(
                size=36,
            )
        )
    )
    
    return fig

In [7]:
timestamp = datetime.now().strftime("%m%d_%H%M%S")

hyper_coefficients = []

for i in range(5):
    values_for_this_param = params_binned[:,i] # ith column
    Fz_medians = [(ub + lb) / 2 for lb, ub in Fz_bins]

    coefficients, _ = curve_fit(PACEJKA_PARAMS_FIT_FNS[i], Fz_medians, values_for_this_param, maxfev=150000, p0=PACEJKA_PARAMS_FIT_GUESS[i])
    hyper_coefficients.append(coefficients)
    
surf_fig = surface_fig(all_data, hyper_coefficients, Fz_bins, 0, -800)

surf_fig.write_html(f"fy_out/figures/fy_sens_surf_({timestamp}).html")

In [8]:
print(hyper_coefficients)
for i in range(5):
    if PACEJKA_PARAMS_FIT_FNS[i] == lin_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz")
    elif PACEJKA_PARAMS_FIT_FNS[i] == quad_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz + {hyper_coefficients[i][2]} * (Fz ** 2)")
    elif PACEJKA_PARAMS_FIT_FNS[i] == exp_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * (Math.E ** ({hyper_coefficients[i][2]} * Fz))")
        

[array([-3.48768508e-01, -3.42768833e-04, -1.32470321e-07]), array([ 5.50922218e-01, -2.44368807e-03, -1.23205368e-06]), array([338.39870577,  -1.97694424]), array([3.55389595e-01, 3.12448972e+01, 1.64059211e-02]), array([-2.33809460e-02, -1.77392228e-04, -1.21225987e-07])]
B = -0.34876850845497015 + -0.00034276883261057686 * Fz + -1.3247032127260662e-07 * (Fz ** 2)
C = 0.5509222176106313 + -0.002443688073927094 * Fz + -1.2320536839299613e-06 * (Fz ** 2)
D = 338.39870577325456 + -1.9769442425898722 * Fz
E = 0.3553895949382016 + 31.244897244705058 * (Math.E ** (0.016405921135532898 * Fz))
F = -0.0233809460004577 + -0.00017739222796836902 * Fz + -1.2122598706268264e-07 * (Fz ** 2)


In [9]:
for i in range(5):
    if PACEJKA_PARAMS_FIT_FNS[i] == lin_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz")
    elif PACEJKA_PARAMS_FIT_FNS[i] == quad_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz + {hyper_coefficients[i][2]} * (Fz ** 2)")
    elif PACEJKA_PARAMS_FIT_FNS[i] == exp_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * (Math.E ** ({hyper_coefficients[i][2]} * Fz))")

B = -0.34876850845497015 + -0.00034276883261057686 * Fz + -1.3247032127260662e-07 * (Fz ** 2)
C = 0.5509222176106313 + -0.002443688073927094 * Fz + -1.2320536839299613e-06 * (Fz ** 2)
D = 338.39870577325456 + -1.9769442425898722 * Fz
E = 0.3553895949382016 + 31.244897244705058 * (Math.E ** (0.016405921135532898 * Fz))
F = -0.0233809460004577 + -0.00017739222796836902 * Fz + -1.2122598706268264e-07 * (Fz ** 2)
